# DriftSense full-session model

## tl;dr

- Trained on **634 usable sessions** from **19 participants**.
- Repeated participant-grouped development selected `activity_only` with `C=0.1`.
- On **193 later sessions**, ROC-AUC was **0.566** (participant-bootstrap 95% CI **0.469–0.670**), precision was **0.303**, recall was **0.451**, and F1 was **0.362** at the development-selected threshold.
- The final JSON artifact matches the Python pipeline to a maximum absolute probability error of **1.11e-16** across shared test vectors.


## Context & Methods

The model predicts the later binary post-session alignment answer from information available after the task session ends. It does not use the answer itself. Candidate feature families and regularization were selected with five repeats of participant-grouped five-fold validation on participant-relative days 1–7. The selected configuration and threshold were evaluated once on later sessions.

### Key Assumptions

- Session rows are the unit of analysis and participant IDs define validation groups.
- Prior-session calibration features use only earlier session outcomes and activity.
- The positive-decision cap is 35% in development data.
- Participant IDs are never predictive inputs.


## Data

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from ml.full_session_model import run_full_session_training

SESSIONS = Path('D:\\1.msc\\DriftSense\\driftsense_merged.csv')
ARTIFACTS = Path('D:\\1.msc\\DriftSense\\ml\\artifacts\\full_session_model')

summary = run_full_session_training(
    sessions_path=SESSIONS,
    output_directory=ARTIFACTS,
    development_days=7,
    repeats=5,
    folds=5,
    max_positive_rate=0.35,
)
quality = summary["data_quality"]
display(pd.DataFrame([{
    "sessions": quality["rows"],
    "participants": quality["participants"],
    "usable_labels": quality["usable_binary_labels"],
    "excluded_uncertain_or_missing": quality["excluded_uncertain_or_missing_labels"],
    "drift_prevalence": quality["drift_prevalence"],
    "duplicate_session_ids": quality["duplicate_session_ids"],
    "overlapping_sessions": quality["overlapping_sessions"],
}]))
display(pd.DataFrame([summary["split"]]))


,sessions,participants,usable_labels,excluded_uncertain_or_missing,drift_prevalence,duplicate_session_ids,overlapping_sessions
0,665,19,634,31,0.302839,0,0


,development_days,development_rows,development_participants,chronological_holdout_rows,chronological_holdout_participants,final_training_rows
0,7,441,19,193,18,634


## Results

In [2]:
tuning = pd.read_csv(ARTIFACTS / "full_session_tuning.csv")
best_tuning = (
    tuning.sort_values(["model", "roc_auc", "brier"], ascending=[True, False, True])
    .groupby("model", as_index=False)
    .first()
)
display(best_tuning[["model", "regularization_c", "roc_auc", "brier", "f1", "prompt_rate"]].sort_values("roc_auc", ascending=False))

comparison = pd.read_csv(ARTIFACTS / "full_session_model_comparison.csv")
holdout = comparison[comparison["evaluation"] == "chronological_known_participant_holdout"]
display(holdout[["model", "regularization_c", "n", "roc_auc", "brier", "accuracy", "precision", "recall", "f1", "prompt_rate"]].sort_values("roc_auc", ascending=False))

display(pd.read_csv(ARTIFACTS / "full_session_calibration.csv"))
display(pd.read_csv(ARTIFACTS / "full_session_coefficients.csv").head(15))
display(pd.DataFrame([summary["chronological_holdout"]]))


,model,regularization_c,roc_auc,brier,f1,prompt_rate
0,activity_only,0.10,0.578842,0.215569,0.122699,0.049887
1,context_activity,0.01,0.576667,0.213198,0.027778,0.006803
2,context_activity_time,0.01,0.568558,0.213952,0.055172,0.009070
4,intended_duration_only,0.10,0.551064,0.216573,0.000000,0.000000
5,participant_calibrated_context_activity,0.01,0.547518,0.216455,0.081081,0.015873
3,context_only,0.01,0.525272,0.217166,0.000000,0.000000
6,task_site_domain_only,3.00,0.506809,0.236241,0.198953,0.113379
7,task_type_only,1.00,0.497683,0.222902,0.000000,0.000000


,model,regularization_c,n,roc_auc,brier,accuracy,precision,recall,f1,prompt_rate
7,task_site_domain_only,3.00,193,0.596313,0.197574,0.715026,0.375000,0.117647,0.179104,0.082902
9,task_type_only,1.00,193,0.592102,0.195954,0.735751,0.000000,0.000000,0.000000,0.000000
15,context_activity,0.01,193,0.587131,0.194226,0.751295,1.000000,0.058824,0.111111,0.015544
19,participant_calibrated_context_activity,0.01,193,0.583955,0.196016,0.766839,0.800000,0.156863,0.262295,0.051813
17,context_activity_time,0.01,193,0.575255,0.193252,0.766839,1.000000,0.117647,0.210526,0.031088
13,activity_only,0.10,193,0.566418,0.198830,0.730570,0.428571,0.058824,0.103448,0.036269
11,context_only,0.01,193,0.563656,0.195695,0.735751,0.000000,0.000000,0.000000,0.000000
5,intended_duration_only,0.10,193,0.532381,0.198572,0.735751,0.000000,0.000000,0.000000,0.000000
3,fixed_timer_prompt_all,NaN,193,0.500000,0.735751,0.264249,0.264249,1.000000,0.418033,1.000000
2,majority_class,NaN,193,0.500000,0.197499,0.735751,0.000000,0.000000,0.000000,0.000000


,sessions,mean_predicted_probability,observed_drift_rate,probability_min,probability_max
0,39,0.189731,0.230769,0.128885,0.224040
1,38,0.260696,0.184211,0.225085,0.306984
2,39,0.331633,0.282051,0.306997,0.355983
3,38,0.393763,0.263158,0.357760,0.423993
4,39,0.475638,0.358974,0.427443,0.591379


,transformed_feature,coefficient,absolute_coefficient
0,keyboard_rate_per_min,-0.228149,0.228149
1,click_rate_per_min,0.180290,0.180290
2,log1p_keyboard_activity_count,-0.178178,0.178178
3,scroll_rate_per_min,-0.177043,0.177043
4,log1p_video_playing_seconds,-0.160586,0.160586
5,tab_switch_rate_per_min,0.145637,0.145637
6,log1p_click_count,0.144544,0.144544
7,log1p_scroll_count,0.137168,0.137168
8,away_share,0.135499,0.135499
9,log1p_idle_seconds,-0.122729,0.122729


,n,prevalence,accuracy,precision,recall,f1,roc_auc,brier,prompt_rate,false_prompt_rate_all_sessions,false_prompt_share_of_prompts,threshold,tn,fp,fn,tp
0,193,0.264249,0.580311,0.302632,0.45098,0.362205,0.566418,0.19883,0.393782,0.274611,0.697368,0.362613,89,53,28,23


## Takeaways

1. Activity provides more grouped-development discrimination than task context alone, but uncertainty across participants remains material.
2. The threshold trades accuracy for recall and must be reported together with its false-positive burden.
3. The final artifact is appropriate for session-end research use. A separate cutoff feature export and validation run are required for an in-session intervention model.
4. The participant-resampled confidence interval and calibration table should accompany any paper claim; a single accuracy value would be misleading for this class balance.
